In [1]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

In [16]:
#!uv pip install datasets
df = pd.read_parquet("hf://datasets/juliensimon/sentry-impact-risk/data/sentry_impact_risk.parquet")
df.head()

,v_infinity_kms,palermo_scale_cum,diameter_km,full_name,absolute_magnitude,last_observation,n_potential_impacts,designation,torino_scale,impact_probability,palermo_scale_max,last_observation_jd,year_range_min,year_range_max
0,14.100000,-0.93,1.300,29075 (1950 DA),17.94,2025-04-28,1,29075,NaN,3.770000e-04,-0.93,2.460794e+06,2880,2880
1,5.991698,-1.40,0.490,101955 Bennu (1999 RQ36),20.63,NaT,157,101955,NaN,5.717000e-04,-1.58,2.459126e+06,2178,2290
2,8.419012,-2.38,0.029,(2008 JL3),25.31,2008-05-09,44,2008 JL3,0.0,1.658148e-04,-2.39,2.454596e+06,2027,2122
3,23.760623,-2.69,0.660,(1979 XB),18.54,1979-12-15,4,1979 XB,0.0,8.515158e-07,-2.99,2.444222e+06,2056,2113
4,1.358027,-2.76,0.037,(2000 SG344),24.79,2000-10-03,300,2000 SG344,0.0,2.743395e-03,-3.11,2.451820e+06,2069,2122


In [15]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2193 entries, 0 to 2192
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   v_infinity_kms       2189 non-null   float64       
 1   palermo_scale_cum    2193 non-null   float64       
 2   diameter_km          2193 non-null   float64       
 3   full_name            2193 non-null   str           
 4   absolute_magnitude   2193 non-null   float64       
 5   last_observation     2192 non-null   datetime64[us]
 6   n_potential_impacts  2193 non-null   int64         
 7   designation          2193 non-null   str           
 8   torino_scale         2191 non-null   float64       
 9   impact_probability   2193 non-null   float64       
 10  palermo_scale_max    2193 non-null   float64       
 11  last_observation_jd  2193 non-null   float64       
 12  year_range_min       2193 non-null   Int64         
 13  year_range_max       2193 non-null   Int64  

In [14]:
df.isna().sum()
#df.dropna()

v_infinity_kms         4
palermo_scale_cum      0
diameter_km            0
full_name              0
absolute_magnitude     0
last_observation       1
n_potential_impacts    0
designation            0
torino_scale           2
impact_probability     0
palermo_scale_max      0
last_observation_jd    0
year_range_min         0
year_range_max         0
dtype: int64

In [33]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'  # Force CPU only
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df_reg = df.copy()
df_reg = df_reg.dropna() 
df_reg['log_impact_probability'] = np.log10(df_reg['impact_probability'])

# Data loading and preprocessing
features = df.select_dtypes(include=['number']).columns.tolist()
features.remove('impact_probability')
X = df_reg[features]
y = df_reg['log_impact_probability']
scaler = StandardScaler()

X_processed = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42)

In [38]:
# Model definition
model = tf.keras.Sequential([
    layers.Input(shape=[len(X.keys())]),
    layers.Dense(64, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(1)
])

# Loss, optimizer, and metrics
optimizer = tf.keras.optimizers.Adam()#.RMSprop(0.001)
model.compile(optimizer=optimizer, loss='mse', metrics=['mae']) #metrics=['mae', 'mse']

# Training
model.fit(X_train, y_train, epochs=10, batch_size=64, verbose=2)

# Evaluation
test_loss, test_mae = model.evaluate(X_test, y_test,batch_size=30, verbose=0)
print(f"Test MSE (Loss): {test_loss:.4f}")
print(f"Test MAE (Mean Absolute Error): {test_mae:.4f}")


Epoch 1/10


28/28 - 1s - 27ms/step - loss: 26.1664 - mae: 4.8520
Epoch 2/10
28/28 - 0s - 3ms/step - loss: 9.4046 - mae: 2.6562
Epoch 3/10
28/28 - 0s - 3ms/step - loss: 3.3940 - mae: 1.3992
Epoch 4/10
28/28 - 0s - 3ms/step - loss: 2.1067 - mae: 1.1485
Epoch 5/10
28/28 - 0s - 2ms/step - loss: 1.5229 - mae: 0.9642
Epoch 6/10
28/28 - 0s - 5ms/step - loss: 1.1307 - mae: 0.8310
Epoch 7/10
28/28 - 0s - 2ms/step - loss: 0.8888 - mae: 0.7315
Epoch 8/10
28/28 - 0s - 2ms/step - loss: 0.7217 - mae: 0.6653
Epoch 9/10
28/28 - 0s - 5ms/step - loss: 0.6023 - mae: 0.6116
Epoch 10/10
28/28 - 0s - 2ms/step - loss: 0.5131 - mae: 0.5603
Test MSE (Loss): 0.4488
Test MAE (Mean Absolute Error): 0.5272
